In [43]:
import jax
device = 'cuda'  # Change to 'cuda' to use GPU
N_CHAINS = 10
N_STEPS_PER_CHAIN = 10_0000
DoF = 120
hidden_units = 4096

jax.config.update("jax_platform_name", device)

print(jax.devices())

from jax import random
import jax.numpy as jnp
import time
from functools import partial


@partial(jax.jit, static_argnames=("prob_fn",))
def mh_chain_with_all_random_nums(random_values, prob_fn, init_pos):
    # Placeholder implementation of mh_chain

    def mh_kernel(carry, random_values):
        position, old_prob = carry
        proposal = position + (2 * random_values[0] - 1)
        proposal_prob = prob_fn(proposal)
        accept_prob = jnp.minimum(1.0, proposal_prob / old_prob)
        accept = random_values[-1] < accept_prob
        new_position = jnp.where(accept, proposal, position)
        new_prob = jnp.where(accept, proposal_prob, old_prob)
        carry = (new_position, new_prob)
        return carry, new_position

    initial_prob = prob_fn(init_pos)
    carry = (init_pos, initial_prob)
    positions, _ = jax.lax.scan(mh_kernel, carry, random_values)
    return positions


sampler = jax.vmap(mh_chain_with_all_random_nums, in_axes=(0, None, 0))


from flax import linen as nn
from jax import numpy as jnp

class DenseParametrized(nn.Module):
    features: int
    α: float = 1.0  # parameter for scaling the weights

    @nn.compact
    def __call__(self, x):
        dense = nn.Dense(features=self.features)
        y = dense(x)
        variable = self.variable("test_collection", "variable_name", lambda: jnp.array(0))
        is_initialized = self.has_variable("test_collection", "variable_name")
        if not is_initialized:
            print("Variable initialized")
            variable.value = jnp.array(0)
        y += variable.value
        variable.value += 1  # Update the variable for demonstration
        print(dense.variables)  # Print the parameters of the Dense layer
        return self.α * y


# Define a simple MLP using Flax Linen
class MLP(nn.Module):
    architecture: list
    hidden_activation: callable = nn.tanh
    α: float = 1.0  # parameter for the final wavefunction

    @nn.compact
    def __call__(self, x):
        for i in range(len(self.architecture) - 1):
            x = nn.Dense(features=self.architecture[i + 1])(x)
            if i < len(self.architecture) - 2:
                x = self.hidden_activation(x)

        return x

    def get_forward_pass(self):
        string = ""
        for i in range(len(self.architecture) - 1):
            string += f"Dense({self.architecture[i]} -> {self.architecture[i + 1]})"
            if i < len(self.architecture) - 2:
                string += f" -> {self.hidden_activation.__name__} ->"

        return string

model = MLP(architecture=[DoF, hidden_units, 1], hidden_activation=nn.tanh, α=1.0)
params = model.init(random.PRNGKey(0), jnp.ones((1, DoF)))

@jax.jit
def prob_fn(x):
    psi = model.apply(params, x).squeeze()
    return jnp.square(psi)


print(model.get_forward_pass())

[CudaDevice(id=0)]
Dense(120 -> 4096) -> tanh ->Dense(4096 -> 1)


In [25]:
beta_model = DenseParametrized(features=2, α=1.0)
beta_params = beta_model.init(random.PRNGKey(0), jnp.ones((1, 5)))
# Pass forward to see the print statement
beta_output = beta_model.apply(beta_params, jnp.ones((1, 5)), mutable=["test_collection"])

{'params': {'kernel': Array([[-0.7387073 ,  0.3374457 ],
       [-0.61614865, -0.38103628],
       [ 0.41116402,  0.57405317],
       [ 0.46691555, -0.02079376],
       [ 0.18908007,  0.34764275]], dtype=float32), 'bias': Array([0., 0.], dtype=float32)}}
{'params': {'kernel': Array([[-0.7387073 ,  0.3374457 ],
       [-0.61614865, -0.38103628],
       [ 0.41116402,  0.57405317],
       [ 0.46691555, -0.02079376],
       [ 0.18908007,  0.34764275]], dtype=float32), 'bias': Array([0., 0.], dtype=float32)}}


In [77]:
from typing import Callable

class CustomDense(nn.Module):
    features: int
    # kernel_init: Callable = nn.initializers.lecun_normal()
    kernel_init: Callable = nn.initializers.constant(0.5)
    bias_init: Callable = nn.initializers.zeros_init()

    @nn.compact
    def __call__(self, inputs):
        kernel = self.param('kernel',
                            self.kernel_init, # Initialization function
                            (inputs.shape[-1], self.features))  # shape info.
        y = jnp.dot(inputs, kernel)
        bias = self.param('bias', self.bias_init, (self.features,))
        y = y + bias
        return y

class MLP_explicit(nn.Module):
    architecture: list
    hidden_activation: Callable = nn.tanh
    kernel_init = nn.initializers.constant(0.5)
    bias_init = nn.initializers.zeros_init()

    @nn.compact
    def __call__(self, x):
        for i in range(len(self.architecture) - 1):
            x = CustomDense(features=self.architecture[i + 1])(x)
            if i < len(self.architecture) - 2:
                x = self.hidden_activation(x)

        return x
    
class MLP(nn.Module):
    architecture: list
    hidden_activation: Callable = nn.tanh
    kernel_init: Callable = nn.initializers.constant(0.5)
    bias_init: Callable = nn.initializers.zeros_init()

    @nn.compact
    def __call__(self, x):
        for i in range(len(self.architecture) - 1):
            x = nn.Dense(features=self.architecture[i + 1], kernel_init=self.kernel_init, bias_init=self.bias_init)(x)
            if i < len(self.architecture) - 2:
                x = self.hidden_activation(x)

        return x

DoF = 4
hidden_units = 5
batch_size = 10

key1, key2 = random.split(random.key(0), 2)
x = random.uniform(key1, (batch_size, DoF))

model_mlp = MLP_explicit(architecture=[DoF, hidden_units, 1])
params = model_mlp.init(key2, x)
y = model_mlp.apply(params, x)

model_mlp_flax = MLP(architecture=[DoF, hidden_units, 1])
params_explicit = model_mlp_flax.init(key2, x)
y_explicit = model_mlp_flax.apply(params_explicit, x)

print('initialized parameters:\n', params)
print('initialized parameters:\n', params_explicit)

assert jnp.allclose(y, y_explicit), "Outputs do not match!"

initialized parameters:
 {'params': {'CustomDense_0': {'kernel': Array([[0.5, 0.5, 0.5, 0.5, 0.5],
       [0.5, 0.5, 0.5, 0.5, 0.5],
       [0.5, 0.5, 0.5, 0.5, 0.5],
       [0.5, 0.5, 0.5, 0.5, 0.5]], dtype=float32), 'bias': Array([0., 0., 0., 0., 0.], dtype=float32)}, 'CustomDense_1': {'kernel': Array([[0.5],
       [0.5],
       [0.5],
       [0.5],
       [0.5]], dtype=float32), 'bias': Array([0.], dtype=float32)}}}
initialized parameters:
 {'params': {'Dense_0': {'kernel': Array([[0.5, 0.5, 0.5, 0.5, 0.5],
       [0.5, 0.5, 0.5, 0.5, 0.5],
       [0.5, 0.5, 0.5, 0.5, 0.5],
       [0.5, 0.5, 0.5, 0.5, 0.5]], dtype=float32), 'bias': Array([0., 0., 0., 0., 0.], dtype=float32)}, 'Dense_1': {'kernel': Array([[0.5],
       [0.5],
       [0.5],
       [0.5],
       [0.5]], dtype=float32), 'bias': Array([0.], dtype=float32)}}}
